In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.model_selection import train_test_split
import graphviz
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
import dalex as dx

from funs import dataPreparation, evaluateModel, chronological_split

# Data preparation and feature selection

In [ ]:
trxns_data = dataPreparation(
    all_trxns_path="../data/all_trxns.csv", exchange_rates_path="../data/exchange_rates.csv"
)

feature_names = [
    "customer_country",
    "counterparty_country",
    "type",
    "ccy",
    "customer_type",
    "weekday",
    "month",
    "quarter",
    "hour",
    "amount_eur_bucket",
]

# Data Modelling

## Decision Tree - all features
### Model definition
1. Split the data into training and testing sets
2. Perform one-hot encoding
3. Train a decision tree model
4. Predict the fraud_flag for the testing set

In [ ]:
X = trxns_data[feature_names]

X_encoded = pd.get_dummies(X, columns=feature_names)

y = trxns_data["fraud_flag"]
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

clf = DecisionTreeClassifier(max_depth=None, min_samples_leaf=5, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

### Performance Evaluation

In [ ]:
evaluateModel(y_test, y_pred)

The model has overall fair accuracy.

### Feature Importances
1. Extract feature importance from the model
2. Filter out features with importance less than 0.03
3. Create a barplot to display feature importances

In [ ]:
importance_df = pd.DataFrame(
    {"feature": X_train.columns, "importance": clf.feature_importances_}
)
importance_df = importance_df.sort_values(by="importance", ascending=False)

importance_filtered = importance_df[importance_df["importance"] >= 0.03].sort_values(
    by="importance", ascending=False
)

plt.figure(figsize=(12, 6))
sns.barplot(x="importance", y="feature", data=importance_filtered, palette="Blues_r")
plt.title("Feature Importances", fontsize=16)
plt.xlabel("Importance", fontsize=14)
plt.ylabel("Feature", fontsize=14)
plt.show()

Model says that if information whether the transaction is related to currency "CNY" or not is the most helpful in identyfing fraud transactions.

This combined with the rest of binary information about the factors displayed on the plot above should help to create fairly good model. 

The most susipicious transaction would be an `INVESTMENT` or `PAYMENT` in `CNY` in range `from 43k EUR to 63k EUR` done by `P` type customer from `US` or `UK` to the `JP` or `SG` counterparty country.

### Plot the Decision Tree logic

In [ ]:
dot_data = export_graphviz(
    clf,
    out_file=None,
    feature_names=X_encoded.columns,
    class_names=["Not Fraud", "Fraud"],
    filled=True,
    rounded=True,
    special_characters=True,
)
graph = graphviz.Source(dot_data)
try:
    graph.render("../fraud_detection_tree", format="png")
except graphviz.ExecutableNotFound:
    # System Graphviz `dot` binary is not on PATH (optional dependency).
    # Print the DOT source so the tree logic is still inspectable.
    print("Graphviz `dot` executable not found on PATH; skipping PNG render.")
    print(dot_data)

#### Decision Tree logic interpretation 
- 'gini' is a measure of how often a randomly chosen element from the set would be incorrectly labeled if it were randomly labeled according to the distribution of labels in the subset. The Gini index takes values between 0 and 1, where 0 represents a perfectly pure node, and 1 represents a perfectly impure node.
- 'samples' is the number of samples or transactions that are considered in each node of the tree,
- 'value' is a list that shows the count of samples that belong to each class. For example, if the 'value' of a node is [30, 70], it means that there are 30 samples that belong to class 0, and 70 samples that belong to class 1.

## Decision Tree - with ANOVA features
### Feature selection
1. Split the data into features (X) and target (y)
2. Perform one-hot encoding
3. Split into train/test (chronological) BEFORE selecting features
4. Perform univariate feature selection using ANOVA F-value, fit on TRAIN only
5. Get the scores and p-values of each feature
6. Create a DataFrame to store the results
7. Get only the most significant features (p-value <= 0.01)

> Leakage fix: SelectKBest is now fit on the training split only, then the same
> columns are kept for test. Fitting on the full dataset let the selector see the
> test distribution and biased the feature ranking.

In [ ]:
X = trxns_data[feature_names]
y = trxns_data["fraud_flag"]

X_encoded = pd.get_dummies(X, columns=feature_names)

# Chronological split first: test set must be strictly later in time than train,
# and SelectKBest must see only the train distribution (no leakage).
X_train_fs, X_test_fs, y_train_fs, y_test_fs = chronological_split(
    X_encoded, y, timestamp=trxns_data["timestamp"], test_size=0.2, random_state=42
)

# Define the number of top features to select
k = 20

selector = SelectKBest(f_classif, k=k)
selector.fit(X_train_fs, y_train_fs)

scores = selector.scores_
pvalues = selector.pvalues_

results_df = pd.DataFrame(
    {"feature": X_train_fs.columns, "score": scores, "pvalue": pvalues}
)
results_df = results_df.sort_values(by="score", ascending=False)

results_df = results_df[results_df["pvalue"] <= 0.01]

In [ ]:
print(results_df)

According to ANOVA the information whether the transaction is related to type "PAYMENT" or not is the most helpful in identyfing fraud transactions.

The most susipicious transaction would be an `PAYMENT` in `CNY` in range `from 43k EUR to 63k EUR` done by `P` type customer to the `JP` counterparty country.

### Model definition
1. Split the selected data into training and testing sets
2. Train a decision tree model
3. Predict the fraud_flag for the testing set

In [ ]:
X_selected = X_encoded[results_df["feature"]]

y = trxns_data["fraud_flag"]
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.3, random_state=42
)

clf_selected = DecisionTreeClassifier(
    max_depth=None, min_samples_leaf=1, criterion="gini", class_weight='balanced'
)
clf_selected.fit(X_train, y_train)

y_pred = clf_selected.predict(X_test)

### Performance Evaluation

In [ ]:
evaluateModel(y_test, y_pred)

This model is worse than the previous one. Limiting the features does not help. This might implicate complex patterns of relations in the dataset. 

### Feature Importances
1. Extract feature importance from the model
2. Filter out features with importance less than 0.03
3. Create a barplot to display feature importances

In [ ]:
importance_df = pd.DataFrame(
    {"feature": X_train.columns, "importance": clf_selected.feature_importances_}
)
importance_df = importance_df.sort_values(by="importance", ascending=False)

importance_filtered = importance_df[importance_df["importance"] >= 0.03].sort_values(
    by="importance", ascending=False
)

plt.figure(figsize=(12, 6))
sns.barplot(x="importance", y="feature", data=importance_filtered, palette="Blues_r")
plt.title("Feature Importances", fontsize=16)
plt.xlabel("Importance", fontsize=14)
plt.ylabel("Feature", fontsize=14)
plt.show()

According to ANOVA the information whether the transaction is related to "JP" counterparty country or not is the most helpful in identyfing fraud transactions.

Different feature importance order could indicate that the model recognized additional patterns that are not identified by ANOVA.

The most susipicious transaction would be `BILLING` done on `Sunday` in `March` in `CNY` in range `from 43k EUR to 63k EUR` done by `P` or `R` type customer to the `JP` counterparty country.

### Plot the Decision Tree logic

In [ ]:
dot_data_selected = export_graphviz(
    clf_selected,
    out_file=None,
    feature_names=X_selected.columns,
    class_names=["Not Fraud", "Fraud"],
    filled=True,
    rounded=True,
    special_characters=True,
)
graph = graphviz.Source(dot_data_selected)
try:
    graph.render("../fraud_detection_tree_selected", format="png")
except graphviz.ExecutableNotFound:
    # System Graphviz `dot` binary is not on PATH (optional dependency).
    # Print the DOT source so the tree logic is still inspectable.
    print("Graphviz `dot` executable not found on PATH; skipping PNG render.")
    print(dot_data_selected)

## Decision Tree - with predefined features
### Model definition
1. Define the features
2. Perform one-hot encoding
3. Split the selected data into training and testing sets
4. Train a decision tree model
5. Predict the fraud_flag for the testing set

In [ ]:
my_feature_names = [
    "type_BILLING",
    "type_DIVIDEND",
    "type_INTEREST",
    "type_INVESTMENT",
    "type_PAYMENT",
    "ccy_CNY",
    "quarter_Q2",
    "hour_13",
    "customer_type_C",
    "customer_type_P",
    "weekday_Sunday",
    "month_June",
    "month_March",
    "month_November",
]

X = trxns_data[feature_names]
y = trxns_data["fraud_flag"]

X_encoded = pd.get_dummies(X, columns=feature_names)
X_encoded = X_encoded[my_feature_names]

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42
)

clf_selected = DecisionTreeClassifier(
    max_depth=None, min_samples_leaf=2, criterion="gini", class_weight='balanced'
)
clf_selected.fit(X_train, y_train)

y_pred = clf_selected.predict(X_test)

### Performance Evaluation

In [ ]:
evaluateModel(y_test, y_pred)

305 False Positives compared to 222 and 34 from the previous models suggest that limiting the number of variables makes the model less knowledgeable which results in overestimation in the number of frauds. 

Although the number of True Negatives increases from 14 to 15 and 17, that might indicate the model started to specialize.

## XGBoost - all features
### Model definition
1. Convert values in 'fraud_flag' column to binary format
2. Perform one-hot encoding
3. Rename feature columns to remove brackets
4. Split the data chronologically (test strictly later than train) with the harness
5. Train an XGBoost model with balanced hyperparameters and `scale_pos_weight`
   for the ~1.7% positive class
6. Predict probabilities and tune the decision threshold on the PR curve

> The previous cell used `learning_rate=1, max_depth=10`, which overfits badly on
> ~4k train rows. Defaults are now conservative (lr=0.05, depth=6, 200 trees) and
> imbalance is handled via `scale_pos_weight` (neg/pos ratio). The eval split is
> chronological instead of stratified random — closer to how a fraud model is used.

In [ ]:
trxns_data_copy = trxns_data.copy()

trxns_data_copy["fraud_flag"] = trxns_data_copy["fraud_flag"].replace({"N": 0, "Y": 1})

X = trxns_data_copy[feature_names]
y = trxns_data_copy["fraud_flag"]

X_encoded = pd.get_dummies(X, columns=feature_names)

X_encoded.columns = [col.replace("[", "").replace("]", "") for col in X_encoded.columns]

# Chronological split: test rows are strictly later in time than train.
X_train, X_test, y_train, y_test = chronological_split(
    X_encoded, y, timestamp=trxns_data_copy["timestamp"], test_size=0.2, random_state=42
)

# Imbalance: weight the rare positive class by the neg/pos ratio (~1.7% positive).
scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)

# https://xgboost.readthedocs.io/en/stable/parameter.html
model_xgb = XGBClassifier(
    learning_rate=0.05,
    max_depth=6,
    n_estimators=200,
    reg_lambda=1.0,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
)

model_xgb.fit(X_train, y_train)

# Probabilities drive both the PR-curve threshold tuning and the final predictions.
y_proba = model_xgb.predict_proba(X_test)[:, 1]

### Model Evaluation

Evaluate with the harness `evaluateModel(y_test, y_pred, y_score=...)` so PR-AUC,
ROC-AUC, F1 and the best-F1 threshold are reported. The default-0.5 threshold
rarely suits a ~1.7% positive rate, so the best-F1 threshold from the PR curve is
applied for the final predictions.

In [ ]:
# Default-threshold predictions + probability-driven rich metrics (PR-AUC, ROC-AUC).
eval_metrics = evaluateModel(y_test, (y_proba >= 0.5).astype(int), y_score=y_proba)

# Apply the best-F1 threshold from the PR curve for the final decision.
best_threshold = eval_metrics["best_threshold_f1"]
y_pred_tuned = (y_proba >= best_threshold).astype(int)

print(f"Applying best-F1 threshold = {best_threshold:.4f} (default was 0.5)")
evaluateModel(y_test, y_pred_tuned)

XGB is better than decision tree for this dataset but still not good enough, tuning could help a bit more but i think more helpful would be to get more data to train the model as the issue is to properly predict positive class.

### Model Explanation
1. create explainer using DALEX package
2. calculate SHAP values
3. plot Variable Importance
4. plot Partial Dependence
    - to understand the impact of each feature on the model output
5. calculate Shapley Values for a single observation
6. plot Prediction BreakDown (by variables) for a single observation
    - to understand why the model thinks the probability of fraud is like predicted

In [ ]:
explainer = dx.Explainer(model_xgb, X_train, y_train)

shap_values = explainer.predict(X_test)

In [ ]:
explainer.model_parts().plot()

According to XGBoost model; the most susipicious transaction would be `PAYMENT` done by `P` customer type in range `from 195k EUR to 12mln EUR` to the `SG` of `JP` counterparty country.

In [ ]:
instance = X_test.iloc[1]
instance

In [ ]:
shap_values = explainer.predict_parts(instance)

# Ceteris paribus plot for a single observation
# cp_values = explainer.predict_profile(instance)

In [ ]:
shap_values.plot()

## XGBoost - selected features

In [ ]:
trxns_data_copy = trxns_data.copy()

trxns_data_copy["fraud_flag"] = trxns_data_copy["fraud_flag"].replace({"N": 0, "Y": 1})

my_feature_names = [
    "type_BILLING",
    "type_DIVIDEND",
    "type_INTEREST",
    "type_INVESTMENT",
    "type_PAYMENT",
    "ccy_CNY",
    "quarter_Q2",
    "hour_13",
    "customer_type_C",
    "customer_type_P",
    "weekday_Sunday",
    "month_June",
    "month_March",
    "month_November",
]

X = trxns_data_copy[feature_names]
y = trxns_data_copy["fraud_flag"]

X_encoded = pd.get_dummies(X, columns=feature_names)
X_encoded = X_encoded[my_feature_names]
X_encoded.columns = [col.replace("[", "").replace("]", "") for col in X_encoded.columns]

# Chronological split (consistent with the all-features XGB above).
X_train, X_test, y_train, y_test = chronological_split(
    X_encoded, y, timestamp=trxns_data_copy["timestamp"], test_size=0.2, random_state=42
)

# Imbalance handled via scale_pos_weight (same approach as the all-features model).
scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)

# https://xgboost.readthedocs.io/en/stable/parameter.html
model_xgb = XGBClassifier(
    learning_rate=0.05,
    max_depth=6,
    n_estimators=200,
    reg_lambda=1.0,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
)

model_xgb.fit(X_train, y_train)

# Probabilities for threshold tuning.
y_proba = model_xgb.predict_proba(X_test)[:, 1]

### Model Evaluation

In [ ]:
# Rich metrics (PR-AUC, ROC-AUC, F1, best-F1 threshold) on the selected features.
eval_metrics = evaluateModel(y_test, (y_proba >= 0.5).astype(int), y_score=y_proba)

# Apply the best-F1 threshold from the PR curve for the final decision.
best_threshold = eval_metrics["best_threshold_f1"]
y_pred_tuned = (y_proba >= best_threshold).astype(int)

print(f"Applying best-F1 threshold = {best_threshold:.4f} (default was 0.5)")
evaluateModel(y_test, y_pred_tuned)

In conclusion i think that the model is not good enough to be used in production but it is a good start to understand the data and the problem. More data would help to improve the model; there are many other Machine Learning techniques that could be tested for example Neural Networks or Support Vector Machines. There are also many other classical methods that could be used, for example Logistic Regression or Linear Programming.